# Olist Sellers Agent

Foco exclusivo na análise dos sellers como clientes da Olist, com métricas de receita, ticket médio, frete e performance por categoria. Este notebook comprova que os dados disponíveis sustentam um agente capaz de orientar receita, eficiência operacional, mix de categorias e cenários acionáveis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')

## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes dos sellers

In [ ]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
products = pd.read_csv(base_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
payments = pd.read_csv(base_path + 'olist_order_payments_dataset.csv')
product_category = pd.read_csv(base_path + 'product_category_name_translation.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('orders', orders.shape)
print('order_items', order_items.shape)
print('products', products.shape)
print('payments', payments.shape)
print('sellers', sellers.shape)

## Preparar tabela mestre e métricas por seller

Junções em duas camadas: itens/pedidos sem duplicar pagamentos; pagamentos agregados por pedido antes do cruzamento com receita do seller.

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_status'] = orders['order_status'].astype('category')

items_products = pd.merge(
    order_items,
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)
items_products = pd.merge(
    items_products,
    sellers[['seller_id', 'seller_label', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
    on='seller_id',
    how='left'
)

seller_items = pd.merge(
    items_products,
    orders[[
        'order_id',
        'customer_id',
        'order_status',
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_customer_date'
    ]],
    on='order_id',
    how='left'
)
seller_items = pd.merge(
    seller_items,
    product_category,
    on='product_category_name',
    how='left'
)

seller_items['delivered'] = seller_items['order_status'] == 'delivered'
seller_items['price'] = pd.to_numeric(seller_items['price'], errors='coerce')
seller_items['freight_value'] = pd.to_numeric(seller_items['freight_value'], errors='coerce')
seller_items['category_label'] = seller_items['product_category_name_english'].fillna(
    seller_items['product_category_name']
)

order_payments = (
    payments.assign(payment_value=pd.to_numeric(payments['payment_value'], errors='coerce'))
    .groupby('order_id', as_index=False)
    .agg(payment_total=('payment_value', 'sum'))
)

delivered_items = seller_items[seller_items['delivered']].copy()

seller_order = (
    delivered_items.groupby(['seller_label', 'seller_id', 'seller_state', 'seller_city', 'order_id'], as_index=False)
    .agg(order_revenue=('price', 'sum'), order_freight=('freight_value', 'sum'))
)
seller_order = pd.merge(seller_order, order_payments, on='order_id', how='left')

seller_metrics = (
    seller_order.groupby(['seller_label', 'seller_id', 'seller_state', 'seller_city'], as_index=False)
    .agg(
        revenue=('order_revenue', 'sum'),
        order_count=('order_id', 'nunique'),
        total_freight=('order_freight', 'sum'),
        total_payment=('payment_total', 'sum'),
        avg_order_value=('order_revenue', 'mean'),
        avg_freight=('order_freight', 'mean'),
        avg_payment_value=('payment_total', 'mean'),
    )
)
seller_metrics['freight_ratio'] = seller_metrics['total_freight'] / seller_metrics['revenue']
seller_metrics['revenue_per_order'] = seller_metrics['revenue'] / seller_metrics['order_count']

seller_category = (
    delivered_items.groupby(['seller_label', 'category_label'], as_index=False)
    .agg(revenue=('price', 'sum'), order_count=('order_id', 'nunique'))
)
seller_category_share = seller_category.copy()
seller_category_share['category_share'] = seller_category_share.groupby('seller_label')['revenue'].transform(
    lambda x: x / x.sum()
)

seller_metrics.head(10)

## Cobertura dos dados para o agente de sellers

Validação de volume e completude antes das visualizações.

In [ ]:
n_items = len(seller_items)
n_delivered_items = len(delivered_items)
n_sellers_active = seller_metrics['seller_label'].nunique()
n_orders_delivered = delivered_items['order_id'].nunique()
n_categories = delivered_items['category_label'].nunique()

coverage = pd.DataFrame({
    'metric': [
        'Itens com seller',
        'Itens com categoria de produto',
        'Itens entregues (delivered)',
        'Itens com categoria traduzida (EN)',
        'Pedidos com pagamento agregado',
    ],
    'pct': [
        seller_items['seller_id'].notna().mean() * 100,
        seller_items['product_category_name'].notna().mean() * 100,
        seller_items['delivered'].mean() * 100,
        seller_items['product_category_name_english'].notna().mean() * 100,
        seller_order['payment_total'].notna().mean() * 100,
    ],
})

print(f'Sellers com pelo menos 1 pedido entregue: {n_sellers_active:,}')
print(f'Pedidos entregues (único order_id): {n_orders_delivered:,}')
print(f'Itens em pedidos entregues: {n_delivered_items:,} / {n_items:,}')
print(f'Categorias distintas: {n_categories}')
print()
display(coverage)
print()
display(seller_metrics[['revenue', 'order_count', 'avg_order_value', 'freight_ratio']].describe())
print()
print('Interpretação:')
print(f'- {n_sellers_active:,} sellers e {n_orders_delivered:,} pedidos entregues permitem segmentação e rankings estáveis.')
print(f'- {n_categories} categorias habilitam performance por mix e cross-sell.')
print('- Métricas de frete, ticket e pagamento estão disponíveis por seller e por pedido.')

## Receita por seller

### Top 10, distribuição e concentração (CDF)

In [ ]:
top_sellers = seller_metrics.sort_values('revenue', ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_sellers, x='revenue', y='seller_label', palette='coolwarm')
plt.title('Top 10 sellers por receita total')
plt.xlabel('Receita total (R$)')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(seller_metrics['revenue'], bins=40, ax=axes[0], color='#4c72b0')
axes[0].set_title('Distribuição de receita por seller')
axes[0].set_xlabel('Receita total (R$)')
axes[0].set_ylabel('Quantidade de sellers')
sns.ecdfplot(seller_metrics, x='revenue', ax=axes[1], color='#d62728')
axes[1].set_title('CDF de receita por seller')
axes[1].set_xlabel('Receita total (R$)')
axes[1].set_ylabel('Proporção acumulada')
plt.tight_layout()
plt.show()

## Ticket médio, frete e volume de pedidos

Scatter com quadrantes (medianas) para comparar sellers de alta/baixa receita por pedido e custo de frete.

In [ ]:
MIN_ORDERS = 10
scatter_df = seller_metrics[seller_metrics['order_count'] >= MIN_ORDERS].copy()
med_ticket = scatter_df['avg_order_value'].median()
med_freight = scatter_df['avg_freight'].median()
order_size = scatter_df['order_count'] / scatter_df['order_count'].max() * 200

plt.figure(figsize=(12, 7))
ax = sns.scatterplot(
    data=scatter_df,
    x='avg_order_value',
    y='avg_freight',
    size=order_size,
    sizes=(40, 300),
    alpha=0.65,
    color='#4c72b0',
    legend=False,
)
ax.axvline(med_ticket, color='gray', linestyle='--', linewidth=1, label=f'Ticket mediano ({med_ticket:.0f})')
ax.axhline(med_freight, color='gray', linestyle=':', linewidth=1, label=f'Frete mediano ({med_freight:.0f})')
ax.set_title(f'Ticket médio x frete médio (sellers com ≥{MIN_ORDERS} pedidos; tamanho = volume)')
ax.set_xlabel('Ticket médio por pedido (R$)')
ax.set_ylabel('Frete médio por pedido (R$)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Performance por categoria

Receita agregada, peso por seller e heatmap seller × categoria.

In [ ]:
category_revenue = (
    seller_category.groupby('category_label', as_index=False)
    .agg(revenue=('revenue', 'sum'), order_count=('order_count', 'sum'))
    .sort_values('revenue', ascending=False)
)
top_categories = category_revenue.head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_categories, x='revenue', y='category_label', palette='viridis')
plt.title('Top 10 categorias por receita total (marketplace)')
plt.xlabel('Receita total (R$)')
plt.ylabel('Categoria')
plt.tight_layout()
plt.show()

top_seller_labels = seller_metrics.sort_values('revenue', ascending=False).head(8)['seller_label']
stack_data = seller_category_share[seller_category_share['seller_label'].isin(top_seller_labels)].copy()
stack_pivot = (
    stack_data.pivot_table(index='seller_label', columns='category_label', values='category_share', fill_value=0)
    .loc[top_seller_labels]
)
top_cats_for_stack = category_revenue.head(8)['category_label'].tolist()
other_cols = [c for c in stack_pivot.columns if c not in top_cats_for_stack]
stack_plot = stack_pivot[top_cats_for_stack].copy()
if other_cols:
    stack_plot['Outras'] = stack_pivot[other_cols].sum(axis=1)

stack_plot.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab20')
plt.title('Mix de receita por categoria — top 8 sellers (100% empilhado)')
plt.xlabel('Seller')
plt.ylabel('Participação na receita do seller')
plt.legend(title='Categoria', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

heatmap_sellers = seller_metrics.sort_values('revenue', ascending=False).head(10)['seller_label']
heatmap_cats = category_revenue.head(12)['category_label']
heatmap_data = seller_category[
    seller_category['seller_label'].isin(heatmap_sellers)
    & seller_category['category_label'].isin(heatmap_cats)
].pivot_table(index='seller_label', columns='category_label', values='revenue', fill_value=0)
heatmap_data = heatmap_data.reindex(index=heatmap_sellers, columns=heatmap_cats, fill_value=0)

plt.figure(figsize=(14, 7))
sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.5)
plt.title('Receita por seller × categoria (top 10 sellers × top 12 categorias)')
plt.xlabel('Categoria')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

## Eficiência operacional e vendas

Rankings de frete/receita e receita por pedido.

In [ ]:
efficiency_base = seller_metrics[seller_metrics['order_count'] >= MIN_ORDERS]

plt.figure(figsize=(12, 6))
low_efficiency = efficiency_base.sort_values('freight_ratio', ascending=False).head(10)
sns.barplot(data=low_efficiency, x='freight_ratio', y='seller_label', palette='mako')
plt.title('Top 10 sellers com maior proporção frete / receita')
plt.xlabel('Frete / Receita')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
revenue_per_order_sorted = efficiency_base.sort_values('revenue_per_order', ascending=False).head(10)
sns.barplot(data=revenue_per_order_sorted, x='revenue_per_order', y='seller_label', palette='rocket')
plt.title('Top 10 sellers por receita média por pedido')
plt.xlabel('Receita média por pedido (R$)')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

## Mix de pagamento por seller

In [ ]:
order_seller_payment = pd.merge(
    seller_order[['seller_label', 'order_id']],
    payments[['order_id', 'payment_type', 'payment_value']],
    on='order_id',
    how='inner',
)
order_seller_payment['payment_value'] = pd.to_numeric(order_seller_payment['payment_value'], errors='coerce')

payments_numeric = payments.copy()
payments_numeric['payment_value'] = pd.to_numeric(payments_numeric['payment_value'], errors='coerce')
payment_mix_market = (
    payments_numeric.groupby('payment_type', as_index=False)
    .agg(payment_value=('payment_value', 'sum'))
)
payment_mix_market['share'] = payment_mix_market['payment_value'] / payment_mix_market['payment_value'].sum()

top5_sellers = seller_metrics.sort_values('revenue', ascending=False).head(5)['seller_label']
payment_mix_top = (
    order_seller_payment[order_seller_payment['seller_label'].isin(top5_sellers)]
    .groupby(['seller_label', 'payment_type'], as_index=False)
    .agg(payment_value=('payment_value', 'sum'))
)
payment_mix_top['share'] = payment_mix_top.groupby('seller_label')['payment_value'].transform(
    lambda x: x / x.sum()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=payment_mix_market, x='payment_type', y='share', ax=axes[0], palette='Blues_d')
axes[0].set_title('Mix de pagamento — marketplace (share por valor)')
axes[0].set_xlabel('Tipo de pagamento')
axes[0].set_ylabel('Participação')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(
    data=payment_mix_top,
    x='seller_label',
    y='share',
    hue='payment_type',
    ax=axes[1],
)
axes[1].set_title('Mix de pagamento — top 5 sellers por receita')
axes[1].set_xlabel('Seller')
axes[1].set_ylabel('Share no seller')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Pagamento', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Segmentação geográfica (seller_state)

In [ ]:
state_metrics = (
    seller_metrics.groupby('seller_state', as_index=False)
    .agg(
        revenue=('revenue', 'sum'),
        seller_count=('seller_label', 'nunique'),
        order_count=('order_count', 'sum'),
        avg_ticket=('avg_order_value', 'mean'),
        freight_ratio=('freight_ratio', 'mean'),
    )
    .sort_values('revenue', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=state_metrics, x='seller_state', y='revenue', ax=axes[0], palette='crest')
axes[0].set_title('Receita total por estado do seller')
axes[0].set_xlabel('UF')
axes[0].set_ylabel('Receita (R$)')

sns.barplot(data=state_metrics, x='seller_state', y='avg_ticket', ax=axes[1], palette='flare')
axes[1].set_title('Ticket médio médio por estado do seller')
axes[1].set_xlabel('UF')
axes[1].set_ylabel('Ticket médio (R$)')
plt.tight_layout()
plt.show()

state_freight_median = state_metrics['freight_ratio'].median()
high_freight_states = state_metrics[state_metrics['freight_ratio'] > state_freight_median].sort_values(
    'freight_ratio', ascending=False
)
print('Estados com frete/receita acima da mediana estadual:')
display(high_freight_states[['seller_state', 'revenue', 'freight_ratio', 'seller_count']].head(8))

## Cenários acionáveis para o agente de sellers

Sellers com ≥10 pedidos entregues; limiares por mediana (ou concentração de categoria ≥60%).

In [ ]:
base = seller_metrics[seller_metrics['order_count'] >= MIN_ORDERS].copy()
q_ticket = base['avg_order_value'].median()
q_freight_ratio = base['freight_ratio'].median()
q_orders = base['order_count'].median()
q_rev_per_order = base['revenue_per_order'].median()

scenario_high_freight_low_ticket = base[
    (base['freight_ratio'] >= q_freight_ratio) & (base['avg_order_value'] <= q_ticket)
].sort_values('freight_ratio', ascending=False)

category_concentration = (
    seller_category_share.groupby('seller_label', as_index=False)['category_share'].max()
    .rename(columns={'category_share': 'max_category_share'})
)
scenario_concentrated = base.merge(category_concentration, on='seller_label', how='left')
scenario_concentrated = scenario_concentrated[scenario_concentrated['max_category_share'] >= 0.6].sort_values(
    'max_category_share', ascending=False
)

scenario_many_orders_low_rev = base[
    (base['order_count'] >= q_orders) & (base['revenue_per_order'] <= q_rev_per_order)
].sort_values('order_count', ascending=False)

scenario_few_orders_high_ticket = base[
    (base['order_count'] < q_orders) & (base['avg_order_value'] >= q_ticket)
].sort_values('avg_order_value', ascending=False)

state_summary = seller_metrics.groupby('seller_state', as_index=False).agg(
    freight_ratio=('freight_ratio', 'mean'),
    revenue=('revenue', 'sum'),
)
state_med = state_summary['freight_ratio'].median()
high_cost_states = state_summary[state_summary['freight_ratio'] > state_med]['seller_state']
scenario_regional = base[base['seller_state'].isin(high_cost_states)].sort_values('freight_ratio', ascending=False)

scenarios = [
    (
        'Alto frete + ticket baixo',
        scenario_high_freight_low_ticket,
        'Otimizar custos logísticos e revisar portfólio (produtos leves / maior valor agregado).',
    ),
    (
        'Receita concentrada em poucas categorias',
        scenario_concentrated,
        'Cross-sell e diversificação de categorias complementares.',
    ),
    (
        'Muitos pedidos, baixa receita por pedido',
        scenario_many_orders_low_rev,
        'Upsell e produtos de maior ticket para elevar receita por pedido.',
    ),
    (
        'Poucos pedidos, alto ticket',
        scenario_few_orders_high_ticket,
        'Ampliar tráfego e conversão mantendo posicionamento premium.',
    ),
    (
        'Seller em UF com frete/receita elevado',
        scenario_regional,
        'Apoio operacional local ou ajuste de política logística regional.',
    ),
]

for title, df, recommendation in scenarios:
    print('=' * 72)
    print(title)
    print(recommendation)
    if df.empty:
        print('(Nenhum seller neste recorte com os limiares atuais.)')
    else:
        cols = ['seller_label', 'seller_state', 'revenue', 'order_count', 'avg_order_value', 'freight_ratio']
        if 'max_category_share' in df.columns:
            cols.append('max_category_share')
        display(df[cols].head(5))
    print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(
    data=base,
    x='avg_order_value',
    y='freight_ratio',
    alpha=0.5,
    ax=axes[0],
    color='#4c72b0',
)
if not scenario_high_freight_low_ticket.empty:
    sns.scatterplot(
        data=scenario_high_freight_low_ticket.head(20),
        x='avg_order_value',
        y='freight_ratio',
        ax=axes[0],
        color='#d62728',
        label='Alto frete + ticket baixo',
    )
axes[0].axvline(q_ticket, color='gray', linestyle='--', linewidth=1)
axes[0].axhline(q_freight_ratio, color='gray', linestyle='--', linewidth=1)
axes[0].set_title('Cenário: frete/receita vs ticket')
axes[0].set_xlabel('Ticket médio (R$)')
axes[0].set_ylabel('Frete / Receita')

sns.scatterplot(
    data=base,
    x='order_count',
    y='revenue_per_order',
    alpha=0.5,
    ax=axes[1],
    color='#4c72b0',
)
if not scenario_many_orders_low_rev.empty:
    sns.scatterplot(
        data=scenario_many_orders_low_rev.head(20),
        x='order_count',
        y='revenue_per_order',
        ax=axes[1],
        color='#d62728',
        label='Volume alto, receita/pedido baixa',
    )
axes[1].axvline(q_orders, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(q_rev_per_order, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Cenário: volume vs receita por pedido')
axes[1].set_xlabel('Pedidos')
axes[1].set_ylabel('Receita por pedido (R$)')
plt.tight_layout()
plt.show()

### Conclusão — dados suficientes para o agente de sellers

Os CSVs de pedidos, itens, pagamentos, produtos, sellers e tradução de categorias permitem construir um agente que:

- **Receita e ticket**: rankings, distribuição e CDF mostram concentração e oportunidade de priorização.
- **Frete e eficiência**: frete médio, ratio frete/receita e scatter ticket × frete expõem sellers com margem pressionada por logística.
- **Categorias**: receita por categoria, mix empilhado e heatmap seller × categoria sustentam recomendações de diversificação e cross-sell.
- **Pagamentos**: mix por seller apoia entendimento de comportamento de checkout e parcelamento.
- **Geografia**: agregados por `seller_state` permitem comparar ticket e custo logístico regional.
- **Cenários**: recortes com limiares estatísticos simulam respostas do agente (otimizar frete, upsell, cross-sell, apoio regional).

Próximo passo natural: expor essas métricas como tools/API do agente conversacional, reutilizando as mesmas junções e agregações definidas neste notebook.